# Práctica 6 — Análisis descriptivo con datos de la ONE

**Estudiante:** Brailin Leonardo Méndez Guillermo  
**Objetivo:** Descargar un CSV oficial publicado en datos.gob.do, limpiarlo y producir estadísticas y visualizaciones reproducibles.


Fuente institucional: [Datos abiertos de la ONE](https://www.one.gob.do/transparencia/datos-abiertos). El código consulta el catálogo oficial de datos.gob.do y selecciona un recurso CSV asociado con la ONE.

In [ ]:
import requests, pandas as pd, io, re

API = "https://datos.gob.do/api/3/action"

def csv_resource_from_package(package_id):
    meta = requests.get(f"{API}/package_show", params={"id": package_id}, timeout=30).json()
    assert meta.get("success"), meta
    resources = meta["result"]["resources"]
    csvs = [r for r in resources if (r.get("format") or "").upper() == "CSV"]
    if not csvs:
        raise RuntimeError("El conjunto no ofrece un recurso CSV")
    return csvs[-1]["url"], meta["result"]["title"]

def read_csv_robust(url):
    raw = requests.get(url, timeout=60).content
    for enc in ("utf-8-sig", "latin-1", "cp1252"):
        try:
            return pd.read_csv(io.BytesIO(raw), encoding=enc, sep=None, engine="python")
        except UnicodeDecodeError:
            pass
    raise RuntimeError("No fue posible detectar la codificación")


In [ ]:
# Buscar conjuntos publicados por la Oficina Nacional de Estadística
resultado = requests.get(f"{API}/package_search", params={"q": '"Oficina Nacional de Estadística"', "rows": 100}, timeout=30).json()
assert resultado.get("success"), resultado
candidatos = []
for paquete in resultado["result"]["results"]:
    if any((r.get("format") or "").upper() == "CSV" for r in paquete.get("resources", [])):
        candidatos.append((paquete["id"], paquete["title"]))
print(f"Conjuntos CSV encontrados: {len(candidatos)}")
candidatos[:10]


In [ ]:
# Seleccionar el primer conjunto CSV disponible; puede cambiar el índice si desea otro tema.
if not candidatos:
    raise RuntimeError("No se encontró un CSV de la ONE en el catálogo. Use la alternativa de carga manual de la celda siguiente.")
package_id, titulo = candidatos[0]
url_csv, titulo = csv_resource_from_package(package_id)
df = read_csv_robust(url_csv)
print("Conjunto:", titulo)
print("Recurso:", url_csv)
print("Dimensiones:", df.shape)
df.head()


In [ ]:
# Limpieza general
df.columns = [re.sub(r'\s+', '_', str(c).strip().lower()) for c in df.columns]
df = df.drop_duplicates().copy()
for c in df.columns:
    if df[c].dtype == 'object':
        normalizada = (df[c].astype(str).str.replace(',', '', regex=False)
                      .str.replace('%', '', regex=False).str.strip())
        convertida = pd.to_numeric(normalizada, errors='coerce')
        if convertida.notna().mean() >= .70:
            df[c] = convertida
print("Nulos por columna:")
display(df.isna().sum().sort_values(ascending=False).head(15))
display(df.describe(include='all').T.head(20))


In [ ]:
import matplotlib.pyplot as plt
numericas = df.select_dtypes(include='number').columns.tolist()
if numericas:
    columna = numericas[0]
    df[columna].dropna().plot(kind='hist', bins=20, figsize=(9,4), title=f'Distribución de {columna}')
    plt.xlabel(columna); plt.grid(alpha=.2); plt.show()
    display(df.nlargest(min(10, len(df)), columna).head(10))
else:
    print("El conjunto seleccionado no contiene columnas numéricas reconocibles.")


### Conclusión
El flujo documenta la fuente, elimina duplicados, normaliza encabezados, identifica datos faltantes y convierte columnas numéricas cuando es razonable. Las estadísticas finales deben interpretarse según el diccionario y la unidad de medida del conjunto seleccionado.